Dataset structure should be:

<pre>
data
    yes
        chair_0_rendered.png
        sofa_2_rendered.png
        ...
    no
        chair_1_rendered.png
        sofa_0_rendered.png
        ...
</pre>

In [27]:
import os, random
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import StratifiedShuffleSplit

import numpy as np
from PIL import Image

In [2]:
class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        img, label = self.subset[i]  # returns PIL image if base_ds.transform is None
        if self.transform:
            img = self.transform(img)
        return img, label

In [3]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /2

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /4

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),             # /8
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  # global average pool
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),              # binary logit
        )

    def forward(self, x):
        x = self.features(x)
        x = self.head(x)
        return x  # raw logits

In [4]:
def run_epoch(loader, train: bool):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.float().unsqueeze(1).to(device)  # shape [B,1], values 0/1

        if train:
            optimizer.zero_grad()

        logits = model(imgs)
        loss = criterion(logits, labels)

        if train:
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            total += labels.numel()

        total_loss += loss.item() * labels.size(0)

    avg_loss = total_loss / max(total, 1)
    acc = total_correct / max(total, 1)
    return avg_loss, acc


In [5]:
def predict_yes_no(image_path: str, model_path: str = "yesno_cnn.pt") -> str:
    """Return exactly 'yes' or 'no' for a single image."""
    ckpt = torch.load(model_path, map_location="cpu")
    image_size = ckpt.get("image_size", 256)
    class_to_idx = ckpt["class_to_idx"]
    idx_to_class = {v: k for k, v in class_to_idx.items()}

    # Rebuild the same TinyCNN
    m = TinyCNN()
    m.load_state_dict(ckpt["model_state"])
    m.eval()

    tfm = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])

    img = Image.open(image_path).convert("RGB")
    x = tfm(img).unsqueeze(0)  # [1,3,H,W]

    with torch.no_grad():
        logit = m(x)
        prob_yes = torch.sigmoid(logit)[0, 0].item()

    # In ImageFolder, 'no' should be class 0 and 'yes' class 1 if folders are named that way.
    # We threshold the probability of class 1 ('yes') at 0.5.
    return "yes" if prob_yes >= 0.5 else "no"

In [6]:
DATA_DIR = "data"
IMAGE_SIZE = 256            # set to 600 if you want full-res (slower!)
BATCH_SIZE = 1
EPOCHS = 10
LR = 1e-3
SEED = 1337

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device (CUDA / Apple MPS / CPU)
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
    else torch.device("cpu")
)
print("Using device:", device)

Using device: cuda


In [7]:
base_ds = datasets.ImageFolder(
    DATA_DIR,
    transform=None,  # defer transforms
)

class_names = base_ds.classes  # should be ["no", "yes"] if folders are named like that
print("Classes:", class_names)

# Train/val split (80/20) that's stable with our seed
num_samples = len(base_ds)
indices = list(range(num_samples))
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

labels = base_ds.targets  # class index for each sample
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(labels)), labels))

# Transforms
train_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(degrees=2),
    transforms.ToTensor(),                # [0,1]
])

val_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

Classes: ['no', 'yes']


In [8]:
train_base = Subset(base_ds, train_idx)
val_base = Subset(base_ds, val_idx)
train_ds = TransformSubset(train_base, train_tfms)
val_ds   = TransformSubset(val_base,   val_tfms)

# DataLoaders
num_workers = 2 if os.name != "nt" else 0  # Windows -> 0 workers is safest
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=(device.type == "cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(device.type == "cuda"))


In [9]:
model = TinyCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [24]:
best_val_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.3f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "model_state": model.state_dict(),
            "class_to_idx": base_ds.class_to_idx,
            "image_size": IMAGE_SIZE,
            "arch": "TinyCNN",
        }, "yesno_cnn.pt")
        print("Saved checkpoint: yesno_cnn.pt")

print("Best val acc:", best_val_acc)

Epoch 01/10 | train loss 0.1147 acc 1.000 | val loss 0.0119 acc 1.000
Saved checkpoint: yesno_cnn.pt
Epoch 02/10 | train loss 0.1106 acc 1.000 | val loss 0.0074 acc 1.000
Epoch 03/10 | train loss 0.1640 acc 0.875 | val loss 0.0185 acc 1.000
Epoch 04/10 | train loss 0.2250 acc 0.875 | val loss 0.0043 acc 1.000
Epoch 05/10 | train loss 0.1464 acc 0.875 | val loss 0.0061 acc 1.000
Epoch 06/10 | train loss 0.0521 acc 1.000 | val loss 0.0128 acc 1.000
Epoch 07/10 | train loss 0.1245 acc 1.000 | val loss 0.0093 acc 1.000
Epoch 08/10 | train loss 0.0679 acc 1.000 | val loss 0.0043 acc 1.000
Epoch 09/10 | train loss 0.0981 acc 1.000 | val loss 0.0045 acc 1.000
Epoch 10/10 | train loss 0.1066 acc 1.000 | val loss 0.0050 acc 1.000
Best val acc: 1.0


In [26]:
print(predict_yes_no("E_grade_pics/chair/chair_35_rendered.png"))
print(predict_yes_no("E_grade_pics/chair/chair_71_rendered.png"))

no
yes


/tmp/ipykernel_13350/2865998317.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(model_path, map_location="cpu")
